# cross-entropy-classification-loss — worked example 2: Cross-entropy with reduction='sum' vs 'mean'

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `cross-entropy-classification-loss`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn.functional as F

## Concept

`F.cross_entropy` defaults to `reduction='mean'` (sum of per-example NLLs divided by batch size). With `reduction='sum'` it returns the raw sum, and with `reduction='none'` it returns the per-example vector. Knowing the denominator is the key to converting between them.

## Worked solution

The goal is to compute the summed cross-entropy by hand and confirm it matches `reduction='sum'`, then recover the mean.

1. **Per-example NLL.** `log_probs = F.log_softmax(logits, dim=-1)` then gather the correct-class log-prob with `log_probs[t.arange(B), labels]`. Negating gives the per-example loss vector `(B,)` — exactly what `reduction='none'` produces.
2. **Sum reduction.** `per_ex.sum()` is the total loss. This is what `F.cross_entropy(logits, labels, reduction='sum')` returns; there is no division here.
3. **Why mean = sum / B.** The default `mean` reduction divides the sum by the number of scored examples (here all `B` of them, since there's no `ignore_index`). So `total / B` must equal `F.cross_entropy(..., reduction='mean')`.

The two asserts pin down both reductions against the PyTorch reference, making the denominator relationship explicit.

In [ ]:
import torch.nn.functional as F

def ce_sum_and_mean(logits, labels):
    B = logits.shape[0]
    log_probs = F.log_softmax(logits, dim=-1)
    per_ex = -log_probs[t.arange(B), labels]
    total = per_ex.sum()
    return total, total / B

t.manual_seed(0)
logits = t.randn(6, 3)
labels = t.tensor([0, 2, 1, 1, 0, 2])
total, mean = ce_sum_and_mean(logits, labels)
ref_sum = F.cross_entropy(logits, labels, reduction='sum')
ref_mean = F.cross_entropy(logits, labels, reduction='mean')
print("sum:", round(total.item(), 6), "ref_sum:", round(ref_sum.item(), 6))
print("mean:", round(mean.item(), 6), "ref_mean:", round(ref_mean.item(), 6))
print("match:", t.allclose(total, ref_sum, atol=1e-5) and t.allclose(mean, ref_mean, atol=1e-5))